# Leverage in Practice: Returns, Risk, and Drawdowns


Leverage means multiplying market exposure using borrowed capital.

It amplifies both gains and losses. In this notebook we test whether higher leverage always improves performance (it does not), and we focus on simple-return based leverage simulation.

We will use real stock data (Microsoft CSV if available) and compare unlevered vs levered growth, risk, and drawdowns.


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:,.6f}'.format
plt.style.use('default')


## 1) Load Historical Price Data

This cell tries common file names (`MSFT.csv`, `msft.csv`) and common close columns (`Close`, `Adj Close`, `MSFT`) to keep the notebook robust.


In [7]:
from pathlib import Path

csv_candidates = [Path('MSFT.csv'), Path('msft.csv')]
csv_path = next((p for p in csv_candidates if p.exists()), None)

if csv_path is None:
    raise FileNotFoundError('Could not find MSFT.csv or msft.csv in current directory.')

raw = pd.read_csv(csv_path)

if 'Date' not in raw.columns:
    raise ValueError('CSV must include a Date column.')

close_col = None
for candidate in ['Close', 'Adj Close', 'MSFT']:
    if candidate in raw.columns:
        close_col = candidate
        break

if close_col is None:
    raise ValueError('CSV must include one of: Close, Adj Close, MSFT.')

data = raw[['Date', close_col]].copy()
data = data.rename(columns={close_col: 'Close'})
data['Date'] = pd.to_datetime(data['Date'], errors='coerce')
data['Close'] = pd.to_numeric(data['Close'], errors='coerce')
data = data.dropna(subset=['Date', 'Close']).sort_values('Date').set_index('Date')

print(f'Loaded: {csv_path.name} (Close source: {close_col})')
print(f'Date range: {data.index.min().date()} -> {data.index.max().date()}')
print(f'Rows: {len(data)}')

data.head()


Loaded: MSFT.csv (Close source: MSFT)
Date range: 2014-10-01 -> 2021-05-28
Rows: 1677


,Close
Date,
2014-10-01,38.880161
2014-10-02,38.761574
2014-10-03,39.041103
2014-10-06,39.041103
2014-10-07,38.566746


## 2) Create Simple and Log Returns


In [8]:
simple_returns = data['Close'].pct_change().dropna()
log_returns = np.log(data['Close'] / data['Close'].shift(1)).dropna()

returns_preview = pd.DataFrame({
    'simple_return': simple_returns,
    'log_return': log_returns
})
returns_preview.head()


,simple_return,log_return
Date,,
2014-10-02,-0.003050,-0.003055
2014-10-03,0.007212,0.007186
2014-10-06,0.000000,0.000000
2014-10-07,-0.012150,-0.012225
2014-10-08,0.027455,0.027085


**Important:** leverage should be applied to **simple returns**.

Simple returns map directly to equity changes for one period. Directly scaling log returns in leveraged portfolio simulations gives misleading results.


## 3) Baseline Leverage Scenario (2x)


In [ ]:
leverage = 2
levered_returns = leverage * simple_returns

growth_unlevered = (1 + simple_returns).cumprod()
growth_levered = (1 + levered_returns).cumprod()

print(f'Leverage factor: {leverage}x')
print(f'Final unlevered growth multiple: {growth_unlevered.iloc[-1]:.4f}')
print(f'Final levered growth multiple:   {growth_levered.iloc[-1]:.4f}')


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(growth_unlevered.index, growth_unlevered, label='Unlevered (1x)')
plt.plot(growth_levered.index, growth_levered, label='Levered (2x)')
plt.title('Leveraged vs Unleveraged Growth')
plt.xlabel('Date')
plt.ylabel('Growth of $1')
plt.legend()
plt.tight_layout()
plt.show()


## 4) Extreme Daily Moves and Leverage Scaling


In [ ]:
max_return = simple_returns.max()
min_return = simple_returns.min()

max_levered = leverage * max_return
min_levered = leverage * min_return

print('Daily extremes (simple returns):')
print(f'  Max daily return (unlevered): {max_return:.2%}')
print(f'  Min daily return (unlevered): {min_return:.2%}')
print()
print(f'With {leverage}x leverage:')
print(f'  Max daily return (levered):   {max_levered:.2%}')
print(f'  Min daily return (levered):   {min_levered:.2%}')


At the daily level, leverage scales gains and losses proportionally.

But compounding is nonlinear, so long-run outcomes become asymmetric: large losses hurt future growth more than equally sized gains help.


## 5) High-Leverage Stress Scenario (7x)


In [ ]:
high_leverage = 7
high_lev_returns_raw = high_leverage * simple_returns

critical_leverage = 1 / abs(min_return)
print(f'Worst daily simple return in sample: {min_return:.2%}')
print(f'Critical leverage threshold 1/|min_return|: {critical_leverage:.2f}x')
print(f'Chosen high leverage: {high_leverage}x')

if high_leverage > critical_leverage:
    print('Condition met: leverage > 1/|min_return|, so raw levered losses can exceed -100%.')
else:
    print('Condition not met in this sample for this leverage level.')


In [ ]:
# Margin closeout / loss cap: equity cannot go below zero in this simplified simulation.
high_lev_returns_capped = np.where(high_lev_returns_raw < -1, -1, high_lev_returns_raw)
high_lev_returns_capped = pd.Series(high_lev_returns_capped, index=simple_returns.index, name='high_lev_capped')

growth_high_lev_capped = (1 + high_lev_returns_capped).cumprod()

print(f'Min raw high-leverage daily return:    {high_lev_returns_raw.min():.2%}')
print(f'Min capped high-leverage daily return: {high_lev_returns_capped.min():.2%}')
print(f'Final high-leverage capped growth:     {growth_high_lev_capped.iloc[-1]:.6f}')


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(growth_unlevered.index, growth_unlevered, label='Unlevered (1x)')
plt.plot(growth_high_lev_capped.index, growth_high_lev_capped, label=f'High Leverage ({high_leverage}x) with -100% cap')
plt.title('High-Leverage Growth with Margin Closeout Cap')
plt.xlabel('Date')
plt.ylabel('Growth of $1')
plt.legend()
plt.tight_layout()
plt.show()


A margin call occurs when account equity falls below required levels.

A margin closeout is forced liquidation by the broker. In practice, brokers enforce risk controls so losses are capped by available capital/collateral rather than allowing unlimited negative equity in a retail account.


## 6) Compare Multiple Leverage Levels


In [ ]:
leverage_levels = [1, 2, 3, 5, 7]
growth_curves = {}

for lev in leverage_levels:
    lev_ret_raw = lev * simple_returns
    lev_ret_capped = np.where(lev_ret_raw < -1, -1, lev_ret_raw)
    lev_ret_capped = pd.Series(lev_ret_capped, index=simple_returns.index)
    growth_curves[lev] = (1 + lev_ret_capped).cumprod()

plt.figure(figsize=(12, 6))
for lev in leverage_levels:
    plt.plot(growth_curves[lev].index, growth_curves[lev], label=f'{lev}x')

plt.title('Cumulative Growth Across Leverage Levels (with -100% daily cap)')
plt.xlabel('Date')
plt.ylabel('Growth of $1')
plt.legend(title='Leverage')
plt.tight_layout()
plt.show()


## Key Insights

- Leverage increases both return and risk.
- Higher leverage increases the probability of ruin (or near-ruin).
- Extreme losses can dominate long-term compounded performance.
- More leverage is **not** always better.
- Simple returns must be used for leverage calculations.
- Using log returns as if they scale linearly with leverage can mislead trading analysis.
